Query Enhancement - Query Decomposition

In [1]:
#libraries
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import RunnableSequence

C:\Users\Dell\AppData\Local\Temp\ipykernel_14068\1287250884.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


loading, chunking and embedding the document

In [2]:
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

building the vectorstore and retriever

In [4]:
embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k":4,"lambda_mult":0.7}
)
retriever

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000262884496A0>, search_type='mmr', search_kwargs={'k': 4, 'lambda_mult': 0.7})

initializing the LLM

In [6]:
llm = init_chat_model("groq:llama-3.1-8b-instant")
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002628A164980>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002628A166120>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

query decomposition prompt

In [11]:
decomposition_prompt = PromptTemplate.from_template("""
You are an helpfull assitant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.
Question: "{question}"
IMPORTANT: RETURN ONLY CREATED SUB-QUESTIONS NO EXPLANATION REGARDING THE SUB-QUESTION GENERATION IS REQUIRED
""")

query decomposition chain

In [12]:
decomposition_chain = decomposition_prompt | llm | StrOutputParser()
decomposition_chain

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='\nYou are an helpfull assitant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.\nQuestion: "{question}"\nIMPORTANT: RETURN ONLY CREATED SUB-QUESTIONS NO EXPLANATION REGARDING THE SUB-QUESTION GENERATION IS REQUIRED\n')
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002628A164980>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002628A166120>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)
| StrOutputParser()

testing out the decomposition chain

In [13]:
query = "How does LangChain use memory and agents compared to CrewAI?"
decomposition_question = decomposition_chain.invoke({"question":query})
print(decomposition_question)

1. What is the architecture of LangChain's memory component?
2. How do agents interact with memory in LangChain?
3. What is the difference between LangChain and CrewAI in terms of agent architecture?
4. Are there any notable similarities in how LangChain and CrewAI utilize memory in their systems?


Final QA prompt for LLM

In [14]:
qa_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.
Context:
{context}
Question: {input}
""")

In [15]:
qa_chain = create_stuff_documents_chain(llm=llm,prompt=qa_prompt)

FINAL RAG PIPELINE LOGIC

In [16]:
def full_query_decomposition_ragpipeline(user_query: str):
    sub_questions_text = decomposition_chain.invoke({"question":user_query})
    sub_questions = [q.strip("-.1234567890.").strip() for q in sub_questions_text.split("\n") if q.strip()]
    results=[]
    for subq in sub_questions:
        docs = retriever.invoke(subq)
        result = qa_chain.invoke({"input":subq,"context":docs})
        results.append(f"Q: {subq}\nA: {result}")

    return "\n\n".join(results)

testing out query decomposition pipeline

In [17]:
query = "How does LangChain use memory and agents compared to CrewAI?"
final_answer = full_query_decomposition_ragpipeline(query)
print("Final Answer:")
print(final_answer)

Final Answer:
Q: What is LangChain's architecture for memory integration?
A: Based on the provided context, LangChain's architecture for memory integration involves using memory modules. Specifically, it mentions two types of memory modules:

1. ConversationBufferMemory: This allows the LLM to maintain awareness of previous conversation turns.
2. ConversationSummaryMemory: This summarizes long interactions to fit within token limits.

These memory modules are designed to be easily combined and reused with other components like retrievers, agents, and chains to build scalable and maintainable LLM applications.

Q: How do agents interact with memory in LangChain?
A: Based on the provided context, LangChain agents use memory in a context-aware manner across steps. This suggests that agents have the ability to store and retrieve information from memory as they execute a sequence of tool invocations to achieve a goal, allowing for dynamic decision-making and branching logic.

Q: What is the